# 08 Regressions

**Owner:** Yair

**What this notebook does:** tests the research question with regressions: how does democracy change around the start of a conflict, is it different for countries that import many weapons, and does it matter who supplied them?

**Reads:** `panel.csv`, `crosswalk_sipri.csv`, `raw/trade-register.csv`, `raw/UcdpPrioConflict_v26_1.xlsx`, `app/site/mapdata.json` (Part F)

**Writes:** `reg_panel.csv`, `reg_eventstudy.csv`, `reg_suppliers.csv`, `reg_results.csv`

The full instructions, background and hints are in **`docs/regression_guide_yair.pdf`** (same part names as here). Work top to bottom.
**Must do:** Part 0, A, C. **Should do:** B, D. **Nice to have:** E, F, G.

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Golden rule: none of these models proves cause and effect. Write "countries that ... tended to ...", never "arms cause ...".

## Setup

Install the new packages once (in a terminal, from the repo root), then add them to `requirements.txt`:

`.venv/bin/pip install statsmodels linearmodels lifelines scipy`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

In [ ]:
panel = pd.read_csv(CLEAN + "panel.csv")
print(panel.shape)
print(panel.head(3))

---
# Part 0: The analysis panel

**Step 1. Sort and log arms.** Sort `panel` by `iso3` and `year`. Add `log_arms = np.log1p(arms_imports_tiv)`
(TIV is very skewed: many zeros, a few huge values).

In [ ]:
panel = panel.sort_values(["iso3", "year"]).reset_index(drop=True)
panel["log_arms"] = np.log1p(panel["arms_imports_tiv"])


**Step 2. Onsets.** Add a column `onset`: 1 in a year with `conflict_active == 1` where the **two years before had no conflict**
(the same rule as 06, Part B Step 1), otherwise 0. Hint: per country, `shift(1)` and `shift(2)` of `conflict_active`.
Also add `war_onset`: 1 if the onset year has `max_intensity == 2`.

In [ ]:
panel = panel.sort_values(["iso3", "year"]).copy()

by_country = panel.groupby("iso3")["conflict_active"]
previous_year = by_country.shift(1)
two_years_before = by_country.shift(2)

panel["onset"] = (
    panel["conflict_active"].eq(1)
    & previous_year.eq(0)
    & two_years_before.eq(0)
).astype(int)

panel["war_onset"] = (
    panel["onset"].eq(1) & panel["max_intensity"].eq(2)
).astype(int)



**Step 3. Lags and windows.** Add, per country:
`arms_pre5` = mean of `log_arms` in the 5 years before (shift(1) then rolling(5)),
`arms_10` = sum of `arms_imports_tiv` over the last 10 years (rolling(10, min_periods=1)),
`dem_lag1` = `democracy_index` one year earlier, and `post1991` = 1 from 1992 on.
Save the table as `reg_panel.csv`.

In [ ]:
g = panel.groupby("iso3", sort=False)
panel["arms_pre5"] = g["log_arms"].transform(lambda s: s.shift(1).rolling(5).mean())
panel["arms_10"] = g["arms_imports_tiv"].transform(lambda s: s.rolling(10, min_periods=1).sum())
panel["dem_lag1"] = g["democracy_index"].shift(1)
panel["post1991"] = (panel["year"] >= 1992).astype(int)
panel.to_csv(CLEAN + "reg_panel.csv", index=False)

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
rp = pd.read_csv(CLEAN + "reg_panel.csv")
for col in ["iso3", "year", "log_arms", "onset", "war_onset", "arms_pre5", "arms_10", "dem_lag1", "post1991"]:
    assert col in rp.columns, "missing column " + col
assert not rp.duplicated(["iso3", "year"]).any()
assert rp["onset"].isin([0, 1]).all()
assert int(rp["onset"].sum()) == 281, "06 found 281 onsets with the same rule; you have " + str(int(rp["onset"].sum()))
assert (rp["log_arms"] >= 0).all()
print("onsets:", int(rp["onset"].sum()), "of which wars:", int(rp["war_onset"].sum()))
print("OK")

---
# Part A: The main finding as a regression (must do)

**Step 1. Outcomes.** For k = 1 to 10 make `y_k = democracy_index(t + k) - democracy_index(t - 1)` per country
(shift(-k) and shift(1)). Standardise `arms_pre5` into `arms_z` (mean 0, sd 1).

In [ ]:
panel = panel.sort_values(["iso3", "year"]).copy()
democracy_by_country = panel.groupby("iso3", sort=False)["democracy_index"]
for k in range(1, 11):
    panel[f"y_{k}"] = democracy_by_country.transform(lambda s: s.shift(-k) - s.shift(1))

panel["arms_z"] = (panel["arms_pre5"] - panel["arms_pre5"].mean()) / panel["arms_pre5"].std()
panel

**Step 2. A1 regression for each k.** `y_k ~ onset + onset:arms_z + arms_z + EntityEffects + TimeEffects` with `PanelOLS`,
data indexed by `["iso3", "year"]`, `cov_type="clustered", cluster_entity=True`. Collect for every k: `b1` (onset),
`b2` (onset:arms_z), their 95% intervals, `n_obs` and `n_countries` in a dataframe `a1`.

In [ ]:
panel_idx = panel.set_index(["iso3", "year"])
rows = []
for k in range(1, 11):
    res = PanelOLS.from_formula(f"y_{k} ~ onset + onset:arms_z + arms_z + EntityEffects + TimeEffects", data=panel_idx).fit(
        cov_type="clustered", cluster_entity=True)
    ci = res.conf_int()
    cov = res.cov
    b1_plus_b2 = res.params["onset"] + res.params["onset:arms_z"]

    cov = res.cov.astype("float64")

    var_sum = (
        float(cov.loc["onset", "onset"])
        + float(cov.loc["onset:arms_z", "onset:arms_z"])
        + 2.0 * float(cov.loc["onset", "onset:arms_z"])
    )
    se_sum = np.sqrt(var_sum)

    # Use the same 95% critical value as res.conf_int()
    critical = (
        (ci.loc["onset", "upper"] - res.params["onset"])
        / res.std_errors["onset"]
    )
    low_sum = b1_plus_b2 - critical * se_sum
    high_sum = b1_plus_b2 + critical * se_sum
    
    rows.append({
        "k": k,
        "b1": res.params["onset"],
        "b2": res.params["onset:arms_z"],
        "low_b1": ci.loc["onset", "lower"],
        "high_b1": ci.loc["onset", "upper"],
        "low_b2": ci.loc["onset:arms_z", "lower"],
        "high_b2": ci.loc["onset:arms_z", "upper"],
        "b1_plus_b2": b1_plus_b2,
        "low_sum": low_sum,
        "high_sum": high_sum,
        "n_obs": res.nobs,
        "n_countries": res.entity_info["total"]
    })
a1 = pd.DataFrame(rows)

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
assert isinstance(a1, pd.DataFrame), "make a dataframe a1"
assert len(a1) == 10, "one row per k = 1 ... 10"
for col in ["k", "b1", "b2", "n_obs"]:
    assert col in a1.columns, "missing column " + col
assert a1["n_obs"].is_monotonic_decreasing or a1["n_obs"].nunique() > 1, "n_obs should shrink as k grows"
print(a1[["k", "b1", "b2", "n_obs"]])
print("OK")

**Step 3. Plot A1.** b1 and b1 + b2 (one sd more arms imports) against k, with 95% bands and a dashed zero line.

In [ ]:
fig, ax = plt.subplots()
ax.plot(a1["k"], a1["b1"], marker="o", label="Average arms importer (b1)")
ax.fill_between(a1["k"], a1["low_b1"], a1["high_b1"], alpha=0.2)

ax.plot(a1["k"], a1["b1_plus_b2"], marker="o", label="1 sd more arms imports (b1 + b2)")
ax.fill_between(a1["k"], a1["low_sum"], a1["high_sum"], alpha=0.2)

ax.axhline(0, color="black", linestyle="--")

ax.set_title("Change in democracy after a conflict onset (A1, country and year fixed effects)")
ax.set(xlabel="Years after conflict onset (k)",
       ylabel="Change in democracy index since t − 1")

ax.legend()
plt.show()

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

In the years after a conflict onset, the democracy index of a country with average arms imports is slightly lower than in other countries in the same year, by about 0.005 to 0.011 points on the 0 to 1 scale. For countries that imported one standard deviation more arms in the five years before the onset, the estimate is slightly positive in the first five years (about +0.001 to +0.011), but from about k = 7 the two lines are almost identical. However, all 95% confidence intervals include zero at every horizon, so we cannot distinguish either estimate from zero. With only 281 onsets and a democracy index that changes slowly, the data are not precise enough to show a clear difference.

**Step 4. A2 event study.** Event-time dummies for t = -10 ... +10 around each onset (bin the ends, leave out t = -1 as reference),
outcome `democracy_index`, same fixed effects and clustering. Plot the 21 coefficients with intervals and a dashed line at t = 0.
Are the leads (t = -10 ... -2) close to zero?

In [ ]:
# A2: event-time dummies around every onset (a country can have several onsets)
panel = panel.sort_values(["iso3", "year"]).copy()
onset_by_country = panel.groupby("iso3")["onset"]

event_cols = []
for j in range(-9, 10):
    if j == -1:
        continue                                  # t = -1 is the reference year
    name = f"ev_m{-j}" if j < 0 else f"ev_p{j}"
    # 1 if this country had an onset j years ago (j < 0: an onset |j| years ahead)
    panel[name] = onset_by_country.shift(j).fillna(0)
    event_cols.append(name)

# binned end points: all onsets 10 or more years ago / 10 or more years ahead
cum = onset_by_country.cumsum()
total = onset_by_country.transform("sum")
panel["ev_p10"] = cum.groupby(panel["iso3"]).shift(10).fillna(0)
panel["ev_m10"] = (total - cum.groupby(panel["iso3"]).shift(-9)).fillna(0)
event_cols = ["ev_m10"] + event_cols + ["ev_p10"]

panel_idx = panel.set_index(["iso3", "year"])
formula = "democracy_index ~ " + " + ".join(event_cols) + " + EntityEffects + TimeEffects"
res_a2 = PanelOLS.from_formula(formula, data=panel_idx).fit(cov_type="clustered", cluster_entity=True)

# collect the coefficients, add t = -1 as 0 (the reference)
ci = res_a2.conf_int()
rows = [{"t": -1, "coef": 0.0, "ci_low": 0.0, "ci_high": 0.0}]
for name in event_cols:
    t = -int(name[4:]) if name.startswith("ev_m") else int(name[4:])
    rows.append({"t": t, "coef": res_a2.params[name],
                 "ci_low": ci.loc[name, "lower"], "ci_high": ci.loc[name, "upper"]})
a2 = pd.DataFrame(rows).sort_values("t").reset_index(drop=True)
print(a2.round(4))

# plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(a2["t"], a2["coef"], marker="o", label="Coefficient")
ax.fill_between(a2["t"], a2["ci_low"], a2["ci_high"], alpha=0.2, label="95% confidence interval")
ax.axhline(0, color="black", linestyle="--")
ax.axvline(0, color="grey", linestyle="--")
ax.set_title("Democracy index around a conflict onset (A2 event study, reference t = -1)")
ax.set(xlabel="Years since conflict onset (t)", ylabel="Democracy index relative to t = -1")
ax.legend()
plt.show()

**Step 5. A3 stacked event study.** For each onset: the onset country for t = -10 ... +10 plus clean controls (no conflict in that
21-year window), one stack id each. Stack them, use stack-by-country and stack-by-year fixed effects, estimate the event dummies
interacted with "treated". Then do it separately for high and low importers (or with arms_pre5). See the guide, Part A3.
Save `reg_eventstudy.csv` with columns `t, group, coef, ci_low, ci_high, n` (group: "all", "high importer", "low importer").

In [ ]:
# A3: stacked event study
# One "stack" per onset: the onset country for t = -10 ... +10, plus clean controls
# (countries with no conflict at all in the same 21 years).
onsets = panel[panel["onset"] == 1][["iso3", "year", "arms_pre5"]]

# high / low importer: arms_pre5 above or below the median of all onsets
median_arms = onsets["arms_pre5"].median()

stacks = []
for stack_id, (iso3, onset_year, arms) in enumerate(onsets.itertuples(index=False)):
    window = panel[panel["year"].between(onset_year - 10, onset_year + 10)]

    # clean controls: no conflict year in this window
    conflict_years = window.groupby("iso3")["conflict_active"].sum()
    clean = conflict_years[conflict_years == 0].index
    keep = window[(window["iso3"] == iso3) | (window["iso3"].isin(clean) & (window["iso3"] != iso3))].copy()

    keep["stack"] = stack_id
    keep["treated"] = (keep["iso3"] == iso3).astype(int)
    keep["t"] = keep["year"] - onset_year
    if pd.isna(arms):
        keep["group"] = "no arms data"
    elif arms > median_arms:
        keep["group"] = "high importer"
    else:
        keep["group"] = "low importer"
    stacks.append(keep[["stack", "iso3", "year", "t", "treated", "group", "democracy_index"]])

stacked = pd.concat(stacks, ignore_index=True)
print("stacks:", stacked["stack"].nunique(), " rows:", len(stacked))

# event dummies, only for the onset country (treated); t = -1 is the reference
event_cols = []
for j in range(-10, 11):
    if j == -1:
        continue
    name = f"tr_m{-j}" if j < 0 else f"tr_p{j}"
    stacked[name] = ((stacked["t"] == j) & (stacked["treated"] == 1)).astype(int)
    event_cols.append(name)

# fixed effects: stack-by-country (entity) and stack-by-year (time)
stacked["stack_iso"] = stacked["stack"].astype(str) + "_" + stacked["iso3"]
stacked["stack_year"] = stacked["stack"] * 10000 + stacked["year"]


def run_stacked(data, group_name):
    """Stacked event study on one set of stacks; returns one row per t."""
    data = data.dropna(subset=["democracy_index"])
    data_idx = data.set_index(["stack_iso", "stack_year"])
    clusters = pd.Series(pd.factorize(data["iso3"])[0], index=data_idx.index, name="country")
    formula = "democracy_index ~ " + " + ".join(event_cols) + " + EntityEffects + TimeEffects"
    res = PanelOLS.from_formula(formula, data=data_idx).fit(cov_type="clustered", clusters=clusters)
    ci = res.conf_int()

    treated = data[data["treated"] == 1]
    rows = []
    for j in range(-10, 11):
        n = int(treated[treated["t"] == j]["stack"].nunique())     # onsets with data at this t
        if j == -1:
            rows.append({"t": j, "group": group_name, "coef": 0.0, "ci_low": 0.0, "ci_high": 0.0, "n": n})
            continue
        name = f"tr_m{-j}" if j < 0 else f"tr_p{j}"
        rows.append({"t": j, "group": group_name, "coef": res.params[name],
                     "ci_low": ci.loc[name, "lower"], "ci_high": ci.loc[name, "upper"], "n": n})
    return pd.DataFrame(rows)


a3 = pd.concat([
    run_stacked(stacked, "all"),
    run_stacked(stacked[stacked["group"] == "high importer"], "high importer"),
    run_stacked(stacked[stacked["group"] == "low importer"], "low importer"),
], ignore_index=True)

a3.to_csv(CLEAN + "reg_eventstudy.csv", index=False)
print(a3[a3["group"] == "all"].round(4))

# plot: all onsets, and high vs low importers
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
part = a3[a3["group"] == "all"]
axes[0].plot(part["t"], part["coef"], marker="o", color="black")
axes[0].fill_between(part["t"], part["ci_low"], part["ci_high"], alpha=0.2, color="grey")
axes[0].set_title("All onsets")
for group, colour in [("high importer", "tab:red"), ("low importer", "tab:blue")]:
    part = a3[a3["group"] == group]
    axes[1].plot(part["t"], part["coef"], marker="o", color=colour, label=group)
    axes[1].fill_between(part["t"], part["ci_low"], part["ci_high"], alpha=0.15, color=colour)
axes[1].set_title("High vs low arms importers")
axes[1].legend()
for ax in axes:
    ax.axhline(0, color="black", linestyle="--")
    ax.axvline(0, color="grey", linestyle="--")
    ax.set_xlabel("Years since conflict onset (t)")
axes[0].set_ylabel("Democracy index relative to t = -1")
fig.suptitle("Stacked event study (A3): onset countries vs clean controls")
plt.tight_layout()
plt.show()


**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
es = pd.read_csv(CLEAN + "reg_eventstudy.csv")
assert list(es.columns) == ["t", "group", "coef", "ci_low", "ci_high", "n"]
assert es["t"].min() == -10 and es["t"].max() == 10
assert "all" in set(es["group"])
assert (es["ci_low"] <= es["coef"]).all() and (es["coef"] <= es["ci_high"]).all()
ref = es[es["t"] == -1]
assert len(ref) == 0 or ref["coef"].abs().max() < 1e-9, "t = -1 is the reference: leave it out or set it to 0"
print(es[es["group"] == "all"].head(12))
print("OK")

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

In the stacked event study, the democracy index of onset countries develops in parallel with clean control countries in the ten years before the onset (all leads between −0.006 and +0.004, with intervals including zero), so there is no sign of a pre-trend. After the onset, the democracy index of onset countries falls behind the controls, by about 0.013 after 4 to 5 years and about 0.018 to 0.025 after 6 to 9 years, where the 95% confidence intervals no longer include zero. Both high and low arms importers show this decline, and the drop is larger for high importers in later years (about −0.03 vs −0.01 at t = 7 to 9), but their intervals overlap heavily, so the difference between the two groups is not clear. The result is consistent with conflict onsets being followed by a small, slow decline in democracy, but other shocks around the onset could explain part of it.

---
# Part B: Does it depend on the kind of case? (should do)

**Step 1. Conflict type at onset.** Read the raw UCDP file as in 02 and attach `type_of_conflict` (1 to 4) of the conflict that
started in each onset year.

In [ ]:
# your code here

**Step 2. Splits.** Run the A1 model for k = 5 and k = 10 with one extra interaction at a time: `war_onset`, conflict type,
`post1991`, starting regime (`regime_type` at t - 1). One table: split, b1, b2, intervals, number of onsets, number of countries.
Groups with fewer than about 15 onsets: report, do not interpret.

In [ ]:
# your code here

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

...

---
# Part C: Who supplied the arms (must do)

**Step 1. Full supplier table.** Rebuild deliveries per recipient, supplier and year from `trade-register.csv` with the cleaning of 04,
but keep every supplier. Map supplier names to iso3 (crosswalk_sipri plus: Soviet Union → RUS, Czechoslovakia → CZE, Yugoslavia → SRB,
East Germany (GDR) → DDR, United States → USA). Drop non-state suppliers. Print the unmatched names and the TIV they cover.

In [ ]:
# your code here

**Step 2. Supplier democracy.** Merge each supplier's `democracy_index` and `regime_type` in the delivery year. Per recipient and year:
`sup_dem` = TIV-weighted mean supplier democracy, `share_dem` = share of TIV from suppliers with regime_type 2 or 3.
Save `reg_suppliers.csv` with columns `iso3, year, sup_dem, share_dem`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
sp = pd.read_csv(CLEAN + "reg_suppliers.csv")
assert list(sp.columns) == ["iso3", "year", "sup_dem", "share_dem"]
assert sp["sup_dem"].between(0, 1).all(), "sup_dem is a democracy index, 0 to 1"
assert sp["share_dem"].between(0, 1).all(), "share_dem is a share, 0 to 1"
assert not sp.duplicated(["iso3", "year"]).any()
print(sp.groupby(sp["year"] < 1992)["sup_dem"].mean())
print("OK")

**Step 3. Regression.** Make `sup_dem_pre5` (5-year pre-window mean) and add `onset:sup_dem_pre5` to the A1 model for k = 5 and 10,
keeping `onset:arms_z`. Only countries with imports in the window.

In [ ]:
# your code here

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

...

**Step 4 (optional). Supplier switching.** Main supplier = most TIV in a 5-year window; switch = it changes. Does a regime change raise
the chance of a switch in the next 5 years? Linear probability model with country and year fixed effects.

In [ ]:
# your code here

---
# Part D: Turn the question around (should do)

**Step 1. What predicts an onset?** Sample: country-years with no conflict in the two previous years.
`onset ~ dem_lag1 + dem_lag1^2 + log_arms(t-1) + regime dummies(t-1)`: (a) linear probability model with country and year fixed effects,
(b) `smf.logit` with year dummies, report average marginal effects. Plot the predicted onset probability over democracy 0 to 1.

In [ ]:
# your code here

**Step 2. Arms before or after?** Run the Part A event study with `log_arms` as the outcome. Same chart format.

In [ ]:
# your code here

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

...

---
# Part E: Survival models (nice to have)

**Step 1. Recovery time.** Per onset: years until democracy is back at its t = -1 level (censored at 2025 or the next onset).
Kaplan-Meier curves for high vs low importers, then a Cox model with arms_pre5, war_onset, regime at t - 1 and post1991 (lifelines).

In [ ]:
# your code here

**Step 2. Risk of autocratisation.** Discrete-time hazard: one row per country-year, event = regime_type drops by one or more.
Logit on conflict_active, log_arms(t-1), dem_lag1 and years-since-last-change dummies.

In [ ]:
# your code here

---
# Part F: Neighbours (nice to have)

**Step 1. Neighbour list** from `../app/site/mapdata.json` (Equal Earth map, 960 x 423): two countries touch if about 10 of their
points are within 1 map unit. Check Germany-France (yes) and Germany-Spain (no).

In [ ]:
# your code here

**Step 2. Spillover.** Share of neighbours with conflict in t - 1 and mean neighbour log_arms; regress own log_arms and the change in
democracy on them (country and year fixed effects).

In [ ]:
# your code here

---
# Part G: GDP and population controls (nice to have)

Download the Maddison Project Database 2023 into `data/raw/`, record it in CONVENTIONS.md, merge on iso3 and year, and add log GDP per
capita and log population (t - 1) to the A1, C3 and D1 models. Do the main results keep sign and size?

In [ ]:
# your code here

---
# Results file and findings

**Step 1. Results table.** One row per model and term: `model, term, coef, se, ci_low, ci_high, n_obs, n_countries`.
Save it as `reg_results.csv`. Then add the four new files to CONVENTIONS.md (made by 08).

**Model names the website reads** (use exactly these in the `model` and `term` columns):

| Part | model | term(s) |
| --- | --- | --- |
| A1 | `A1_k1` ... `A1_k10` | `onset`, `onset:arms_z` |
| B | `B_war_k5`, `B_type_k5`, `B_post1991_k5`, `B_regime_k5` | the interaction, e.g. `onset:war_onset`, `onset:intrastate`, `onset:post1991`, `onset:democracy_t1` |
| C3 | `C3_k5`, `C3_k10` | `onset:sup_dem_pre5` |
| D1 | `D1_lpm` | `dem_lag1`, `dem_lag1_sq`, `log_arms_lag1` |

Other models can be added with any name; the website lists them in a general table.
Until your files exist in `data/clean/`, the website shows placeholder numbers from `data/demo/` with a warning banner.

In [ ]:
# your code here

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
rr = pd.read_csv(CLEAN + "reg_results.csv")
assert list(rr.columns) == ["model", "term", "coef", "se", "ci_low", "ci_high", "n_obs", "n_countries"]
assert len(rr) > 0 and rr["se"].ge(0).all()
print(rr["model"].unique())
print("OK")

## Findings (write in your own words)

1. Main result (Part A):
2. Does arms importing matter (A, B)?
3. Does the supplier matter (C)?
4. What the reverse checks show (D):
5. Limits: endogeneity, small groups, historical states, TIV is not money.

Then tell Nicolas that `reg_eventstudy.csv` and `reg_suppliers.csv` exist, for the website.